### LLM Hallucination Detection

LLMs can sometimes generate information that sounds plausible but is factually incorrect, a phenomenon known as 'hallucination'. This project will create a very basic system to detect potential hallucinations by comparing LLM responses against a small set of known facts (ground truth).

#### Step 1 – Project Goal

The goal is to build a simple mechanism to check if an LLM's response contains information that contradicts or is missing from our known set of facts. This is a rudimentary way to flag potential factual errors.

#### Step 2 – Install libraries if needed

No external libraries need to be installed for this project, as we'll use `pandas` and built-in Python features.

#### Step 3 – Import libraries

We will import the `pandas` library to help us manage our data effectively.

In [1]:
import pandas as pd # Import pandas for data table operations.

This code imports `pandas` as `pd`, which is necessary for creating and manipulating DataFrames, our tabular data structure.

#### Step 4 – Input data

We'll define a small set of prompts, corresponding LLM responses, and a `ground_truth_facts` dictionary that holds known factual information. This will allow us to check for consistency.

In [2]:
# Define our known facts (ground truth)
ground_truth_facts = {
    'Eiffel Tower': 'located in Paris, France, and is 330 meters tall.',
    'Mount Everest': 'the highest mountain in the world, located in the Himalayas.',
    'Mars': 'often called the Red Planet, is the fourth planet from the Sun.'
}

# Define a small dataset of LLM prompts and responses
data = [
    {
        'prompt': 'Tell me about the Eiffel Tower.',
        'llm_response': 'The Eiffel Tower is in Paris, France. It is about 330 meters tall.'
    },
    {
        'prompt': 'Where is Mount Everest?',
        'llm_response': 'Mount Everest is the tallest mountain in the Alps, in Europe.' # Incorrect information
    },
    {
        'prompt': 'Describe Mars.',
        'llm_response': 'Mars is also known as the Green Planet and is the fifth planet from the Sun.' # Incorrect information
    },
    {
        'prompt': 'What about the Eiffel Tower?',
        'llm_response': 'The Eiffel Tower is a famous landmark in Europe.' # Missing key details
    }
]

# Convert the data into a pandas DataFrame
df_hallucination = pd.DataFrame(data) # Create a DataFrame from our LLM responses.

Here, we've set up `ground_truth_facts` as a dictionary, mapping topics to their correct information. We also created a DataFrame `df_hallucination` with various LLM responses, some of which contain factual errors or missing details, to simulate real-world scenarios.

#### Step 5 – Main logic (small loop)

We will iterate through each LLM response and try to detect hallucinations by checking if the response aligns with our `ground_truth_facts`. Our detection logic will be simple: check for contradictions or missing key facts.

In [3]:
hallucination_results = [] # List to store our hallucination detection findings.

# Loop through each LLM response in our DataFrame
for index, row in df_hallucination.iterrows(): # Go through each row of the DataFrame.
    prompt = row['prompt'] # Get the prompt.
    response = row['llm_response'] # Get the LLM's response.
    hallucination_detected = False # Flag to indicate if a hallucination is found, starts as False.
    detection_reason = "No hallucination detected." # Default reason.

    # Simple logic to infer the topic from the prompt (very basic)
    topic = None
    if 'Eiffel Tower' in prompt:
        topic = 'Eiffel Tower'
    elif 'Mount Everest' in prompt:
        topic = 'Mount Everest'
    elif 'Mars' in prompt:
        topic = 'Mars'

    if topic and topic in ground_truth_facts: # If a topic is identified and we have facts for it.
        known_fact = ground_truth_facts[topic].lower() # Get the known fact and convert to lowercase.
        response_lower = response.lower() # Convert the response to lowercase for comparison.

        # Check for contradictions or missing information
        if topic == 'Mount Everest':
            if 'alps' in response_lower or 'europe' in response_lower: # Known contradiction for Everest
                hallucination_detected = True
                detection_reason = "Contains contradictory information (Alps/Europe for Mount Everest)."
            elif 'himalayas' not in response_lower and 'highest mountain' not in response_lower: # Missing key facts
                 hallucination_detected = True
                 detection_reason = "Missing key factual details (Himalayas, highest mountain)."

        elif topic == 'Mars':
            if 'green planet' in response_lower or 'fifth planet' in response_lower: # Known contradictions for Mars
                hallucination_detected = True
                detection_reason = "Contains contradictory information (Green Planet/fifth planet for Mars)."
            elif 'red planet' not in response_lower and 'fourth planet' not in response_lower: # Missing key facts
                hallucination_detected = True
                detection_reason = "Missing key factual details (Red Planet, fourth planet)."

        elif topic == 'Eiffel Tower':
            # Check if key facts are NOT present in the response for Eiffel Tower
            if 'paris' not in response_lower or 'france' not in response_lower or '330 meters' not in response_lower:
                hallucination_detected = True
                detection_reason = "Missing key factual details (e.g., Paris, France, height)."

    # If no specific topic rule, or no hallucination detected by rules, but response is very short/vague
    if not hallucination_detected and len(response.split()) < 10 and topic:
        # This is a very simple check for overly brief/vague responses which might imply missing info
        hallucination_detected = True
        detection_reason = "Response is too vague or lacks sufficient detail for the topic."

    # Store the results
    hallucination_results.append({
        'prompt': prompt,
        'llm_response': response,
        'topic_checked': topic if topic else 'N/A',
        'ground_truth_fact': ground_truth_facts.get(topic, 'N/A') if topic else 'N/A',
        'hallucination_detected': hallucination_detected,
        'detection_reason': detection_reason
    })

# Convert the results into a DataFrame
detection_df = pd.DataFrame(hallucination_results) # Create a DataFrame from our detection results.

This code block iterates through each LLM response. It first tries to identify the topic of the prompt. Then, it uses simple `if` statements and keyword checks to compare the LLM's response against the `ground_truth_facts`. If it finds a contradiction (like 'Alps' for Mount Everest) or a significant piece of factual information is missing, it flags the response as potentially `hallucination_detected` and provides a `detection_reason`. This is a very basic form of factual consistency checking.

#### Step 6 – Output results

We will display the results of our hallucination detection and save them to a CSV file.

In [4]:
print("\nLLM Hallucination Detection Results:") # Print a header.
display(detection_df) # Display the DataFrame with detection results.

# Save the detection results to a CSV file
output_filename = 'llm_hallucination_detection.csv' # Name for the output CSV file.
detection_df.to_csv(output_filename, index=False) # Save the DataFrame, excluding the pandas index.

print(f"\nDetection results saved to {output_filename}") # Confirmation.


LLM Hallucination Detection Results:


,prompt,llm_response,topic_checked,ground_truth_fact,hallucination_detected,detection_reason
0,Tell me about the Eiffel Tower.,"The Eiffel Tower is in Paris, France. It is ab...",Eiffel Tower,"located in Paris, France, and is 330 meters tall.",False,No hallucination detected.
1,Where is Mount Everest?,Mount Everest is the tallest mountain in the A...,Mount Everest,"the highest mountain in the world, located in ...",True,Contains contradictory information (Alps/Europ...
2,Describe Mars.,Mars is also known as the Green Planet and is ...,Mars,"often called the Red Planet, is the fourth pla...",True,Contains contradictory information (Green Plan...
3,What about the Eiffel Tower?,The Eiffel Tower is a famous landmark in Europe.,Eiffel Tower,"located in Paris, France, and is 330 meters tall.",True,"Missing key factual details (e.g., Paris, Fran..."



Detection results saved to llm_hallucination_detection.csv


This code displays the `detection_df`, showing for each prompt and response whether a hallucination was detected and why. It then saves this information to a CSV file, `llm_hallucination_detection.csv`, allowing for easy review and further analysis.

#### Step 7 – Short explanation of the result

Looking at the `detection_df`, you can see that responses containing incorrect facts (like Mount Everest in the Alps or Mars as the Green Planet) or missing crucial details (Eiffel Tower response) are flagged as `hallucination_detected`. This very basic system demonstrates how you might begin to programmatically identify factual inaccuracies in LLM outputs by cross-referencing them with known information. While simple, it highlights the core idea of validating LLM generations.